# 15 · Greedy

> A ogni passo fai la scelta localmente migliore e non torni indietro. Funziona solo se riesci a convincerti (con un controesempio mancato o uno scambio) che la scelta locale non preclude l'ottimo. Quando non funziona, di solito serve la DP.
> Idee per i singoli problemi: [Pattern algoritmici › Greedy](../Fundamentals/Pattern%20algoritmici.md#greedy)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Maximum Subarray | Kadane: riparti da zero se la somma corrente è negativa | `cur = max(x, cur + x)` |
| Jump Game | tieni il punto più lontano raggiungibile | `reach = max(reach, i + nums[i])`, `if i > reach` |
| Jump Game II | BFS "a finestre": ogni salto copre un intervallo di indici | `cur_end`, `farthest`, contatore |
| Gas Station | se il totale è ≥ 0 una soluzione esiste; riparti dopo ogni deficit | `sum(gas) - sum(cost)`, `zip` |
| Hand of Straights | parti sempre dalla carta più piccola rimasta | `Counter`, `sorted(counts)` o min-heap |
| Merge Triplets to Form Target Triplet | scarta le triple con un valore oltre il target, unisci le altre | `any`, `all`, set degli indici soddisfatti |
| Partition Labels | ultima occorrenza di ogni carattere | `{ch: i for i, ch in enumerate(s)}` |
| Valid Parenthesis String | intervallo `[lo, hi]` di aperte possibili | `max(lo - 1, 0)`, `hi < 0` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| ultima posizione di ogni elemento | `last = {x: i for i, x in enumerate(xs)}` |
| prima posizione | `first = {}` · `first.setdefault(x, i)` |
| tutti / almeno uno | `all(cond for ...)`, `any(cond for ...)` |
| chiavi ordinate di un conteggio | `for x in sorted(counts):` |
| scorrere due liste insieme | `for g, c in zip(gas, cost):` |
| clamp a zero | `max(value, 0)` |

## Kadane: miglior sotto-array contiguo

La somma migliore che **finisce in `i`** è `x` da solo oppure `x` aggiunto alla migliore che finisce in `i - 1`. Se quella precedente è negativa, ti conviene ripartire.

In [1]:
daily_profit = [-2, 1, -3, 4, -1, 2, 1, -5, 4]

cur = best = daily_profit[0]
for x in daily_profit[1:]:
    cur = max(x, cur + x)
    best = max(best, cur)
print(best)  # -> 6 ([4, -1, 2, 1])

6


⚠️ Inizializza con il primo elemento, non con 0: se tutti i valori sono negativi la risposta è il meno negativo.

## Il più lontano raggiungibile

Tieni il massimo indice raggiungibile finora. Se l'indice corrente lo supera, sei bloccato. Per contare il **numero minimo di salti**, ragiona a finestre: tutti gli indici fino a `cur_end` si raggiungono con `jumps` salti; quando arrivi a `cur_end`, devi fare un salto in più e la nuova finestra arriva a `farthest`.

In [2]:
max_step = [2, 3, 1, 1, 4]

reach = 0
for i, step in enumerate(max_step):
    if i > reach:
        print("bloccato in", i)
        break
    reach = max(reach, i + step)
print(reach >= len(max_step) - 1)  # -> True

jumps, cur_end, farthest = 0, 0, 0
for i in range(len(max_step) - 1):  # l'ultimo indice non richiede un salto
    farthest = max(farthest, i + max_step[i])
    if i == cur_end:
        jumps += 1
        cur_end = farthest
print(jumps)  # -> 2

True
2


## Bilancio che va in negativo

Se percorri una sequenza accumulando un saldo e questo diventa negativo in `i`, nessun punto di partenza tra l'inizio corrente e `i` può funzionare: riparti da `i + 1`. Se il saldo **totale** è negativo, nessuna partenza funziona.

In [3]:
supply = [1, 2, 3, 4, 5]
demand = [3, 4, 5, 1, 2]
print(sum(supply) - sum(demand))  # -> 0: una soluzione esiste

balance, start = 0, 0
for i, (s, d) in enumerate(zip(supply, demand)):
    balance += s - d
    if balance < 0:
        balance, start = 0, i + 1
print(start)  # -> 3

0
3


## Parti dal più piccolo

Quando devi formare gruppi di valori consecutivi, il valore più piccolo rimasto **deve** iniziare un gruppo (non può stare in mezzo a nessun altro). Conta con `Counter` e scorri le chiavi ordinate.

In [4]:
from collections import Counter

sizes = [1, 2, 3, 6, 2, 3, 4, 7, 8]
group = 3
print(len(sizes) % group)  # -> 0: altrimenti impossibile

counts = Counter(sizes)
for x in sorted(counts):
    need = counts[x]
    if need == 0:
        continue
    for y in range(x, x + group):
        if counts[y] < need:  # Counter restituisce 0 per le chiavi mancanti
            print("impossibile da", x)
            break
        counts[y] -= need
print(+counts)  # -> Counter(): il + unario elimina gli zeri

0
Counter()


## Ultima occorrenza

Un dict comprehension con `enumerate` sovrascrive la chiave a ogni occorrenza: alla fine contiene l'**ultimo** indice. Utile per sapere fin dove un gruppo deve estendersi.

In [5]:
s = "ababcbacadefegde"
last = {ch: i for i, ch in enumerate(s)}
print(last["a"], last["e"])  # -> 8 15

end = 0
for i, ch in enumerate(s):
    end = max(end, last[ch])
    if i == end:
        print("fine pezzo in", i)  # -> fine pezzo in 8
        break

8 15
fine pezzo in 8


## Filtrare e combinare con `all`/`any`

In [6]:
target = (2, 7, 5)
triplets = [(2, 5, 3), (1, 8, 4), (1, 7, 5)]

usable = [t for t in triplets if all(v <= goal for v, goal in zip(t, target))]
print(usable)  # -> [(2, 5, 3), (1, 7, 5)]: (1, 8, 4) supera il target

matched = {i for t in usable for i, v in enumerate(t) if v == target[i]}
print(matched == {0, 1, 2})  # -> True

[(2, 5, 3), (1, 7, 5)]
True


## Un intervallo di possibilità

Quando un simbolo jolly può valere più cose, invece di provare tutte le combinazioni tieni il **minimo e il massimo** valore possibile dello stato. Esempio: numero di parentesi aperte con `*` che può essere `(`, `)` o niente.

In [7]:
lo = hi = 0
for ch in "(*))":
    if ch == "(":
        lo, hi = lo + 1, hi + 1
    elif ch == ")":
        lo, hi = lo - 1, hi - 1
    else:  # "*"
        lo, hi = lo - 1, hi + 1
    if hi < 0:
        print("troppe chiuse")
        break
    lo = max(lo, 0)  # non si può scendere sotto zero aperte
print(lo, hi, lo == 0)  # -> 0 0 True

0 0 True


## Errori comuni

- Applicare un greedy senza verificarlo: cerca un controesempio piccolo (es. monete `[1, 3, 4]` e importo 6).
- Kadane inizializzato a 0 con input tutto negativo.
- Jump Game II: iterare fino all'ultimo indice incluso e contare un salto in più.
- Modificare il `Counter` mentre iteri direttamente su di esso: itera su `sorted(counts)` (una lista nuova).